\begin{equation*}
Input: X+ϵ → ℜ^{1x8};ϵ \sim η(\mu,σ)
\end{equation*}

\begin{equation*}
Output: Y=X→ ℜ^{1x8}
\end{equation*}

\begin{equation*}
Flujos: F→ ℜ^{1x59}
\end{equation*}

\begin{equation*}
Costos: C→ ℜ^{1x59}
\end{equation*}

\begin{equation*}
 F_{d} \epsilon F → ℜ^{1x8}
\end{equation*}
\begin{equation*}
 F_{w} \epsilon F → ℜ^{1x1}
\end{equation*}
\begin{equation*}
 C_{d} \epsilon C → ℜ^{1x8}
\end{equation*}
\begin{equation*}
 C_{w} \epsilon C → ℜ^{1x1}
\end{equation*}
\begin{equation*}
 loss: 1-\frac{1}{N}∑exp(-\frac{(F_{d}C_{d}+F_{w}C_{w}-YC_{d})^{2}}{2σ^{2}})
\end{equation*}
variables que conozco, X, Y y C lo demás se estima por la red

Cd tiene magnitudes de 1000000, Cw de 5000

In [2]:
from google.colab import drive
from tensorflow.python.ops.nn_ops import softmax
from keras import backend as k
import numpy as np
from matplotlib.gridspec import GridSpec
from IPython import display
from scipy.optimize import linprog
from sklearn.metrics import mean_squared_error,mean_absolute_error,mean_absolute_percentage_error
import numpy as np
import matplotlib. pyplot as plt
from keras.layers import Activation,Dense,Input,BatchNormalization,Dropout,Conv1D,Flatten,MaxPool1D,Dot,Reshape,Conv2D,Concatenate,ReLU,Lambda,MaxPooling2D,Normalization
import cvxpy as cp
import tensorflow as tf
from keras import Model
from IPython import display
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import warnings
import os
import pandas as pd
import pandas as pd
from IPython import display
path='/content/drive/Shareddrives/red_gas_col/Data/'
from keras.models import Sequential
mse=mean_squared_error
mae=mean_absolute_error
mape=mean_absolute_percentage_error
warnings.filterwarnings("ignore")
#drive.mount('/content/drive')
from google.colab import drive
drive.mount('/content/drive')
import scipy.io as sio
def load_data(path,Tip):
    ind1=[]
    ind2=[]
    for k,j in enumerate([8,9,10,11,12,13,14,15,18]):
        ind1.append([])
        ind2.append([])
        for i in range(1,6):
            ind1[k].append(np.where(sio.loadmat(path+Tip+'Wey'+str(j)+str(i)+'.mat')['inputs']<1e-4)[1])
            ind2[k].append(np.where(sio.loadmat(path+Tip+'Wey'+str(j)+str(i)+'.mat')['inputs']>=1e-4)[1])
    for k,j in enumerate([8,9,10,11,12,13,14,15,18]):
        aux=[]
        aux2=[]
        aux3=[]
        aux4=[]
        aux5=[]
        aux6=[]
        aux7=[]
        aux8=[]
        for i in range(1,6):
            a=sio.loadmat(path+'Inputs/'+'Input1'+str(j)+str(i)+'.mat')['inputs'][ind1[k][i-1]]
            b=sio.loadmat(path+'Inputs/'+'Input2'+str(j)+str(i)+'.mat')['inputs']
            c=sio.loadmat(path+Tip+'Com'+str(j)+str(i)+'.mat')['inputs'].reshape(-1,)[ind1[k][i-1]]
            d=sio.loadmat(path+Tip+'Bal'+str(j)+str(i)+'.mat')['inputs'].reshape(-1,)[ind1[k][i-1]]
            e=sio.loadmat(path+Tip+'Wey'+str(j)+str(i)+'.mat')['inputs'].reshape(-1,)[ind1[k][i-1]]
            f=sio.loadmat(path+'Costos/'+'Cost'+str(j)+str(i)+'.mat')['inputs'].reshape(-1,)[ind1[k][i-1]]
            g=sio.loadmat(path+'Flujos/'+'Flujos'+str(j)+str(i)+'.mat')['inputs'][ind1[k][i-1]]
            h=sio.loadmat(path+'Presiones/'+'Presiones'+str(j)+str(i)+'.mat')['inputs'][ind1[k][i-1]]
            aux.append(a)
            aux2.append(b)
            aux2.append(sio.loadmat(path+'Inputs/'+'Input1'+str(j)+str(i)+'.mat')['inputs'][ind2[k][i-1]])
            aux3.append(c)
            aux4.append(d)
            aux5.append(e)
            aux6.append(f)
            aux7.append(g)
            aux8.append(h)
        aux=np.concatenate(aux)
        all_empty = all(arr.size == 0 for arr in aux2)
        if all_empty:
            aux2 = np.array([])
        else:
            aux2 = np.concatenate([arr for arr in aux2 if arr.size > 0])
        aux3=np.concatenate(aux3)
        aux4=np.concatenate(aux4)
        aux5=np.concatenate(aux5)
        aux6=np.concatenate(aux6)
        aux7=np.concatenate(aux7)
        aux8=np.concatenate(aux8)
        np.save(path+Tip+'Red'+str(j)+'Input1',aux)
        inp=aux
        np.save(path+Tip+'Red'+str(j)+'Input2',aux2)
        np.save(path+Tip+'Red'+str(j)+'Com',aux3)
        np.save(path+Tip+'Red'+str(j)+'Bal',aux4)
        np.save(path+Tip+'Red'+str(j)+'Wey',aux5)
        np.save(path+Tip+'Red'+str(j)+'Cost',aux6)
        np.save(path+Tip +'Red'+str(j)+'Flujos',aux7)
        np.save(path+Tip+'Red'+str(j)+'Presiones',aux8)
    for i in ['Bal', 'Wey', 'Com', 'Cost','Flujos','Presiones']:
        aux = []
        for idx, j in enumerate([8, 9, 10, 11, 12, 13, 14, 15, 18]):
            aux.append([])
            a = np.load(path+Tip+ 'Red' + str(j) + i + '.npy')
            for k in range(np.array(a).shape[0]):
                aux[idx].append(np.array(a[k]))  # Usar idx que corresponde al índice correcto de cada sublista en aux
        sio.savemat(path+Tip+i+'_P'+'.mat', mdict={'inputs': aux})
    aux=[]
    cont=0
    for j in [8,9,10,11,12,13,14,15,18]:
        aux.append([])
        a=np.load(path+Tip+'Red'+str(j)+'Input1.npy')
        for k in range(np.array(a).shape[0]):
            aux[cont].append(np.array(a[k]))
        cont+=1
    sio.savemat(path+Tip+'inputs_P.mat',mdict={'inputs':aux})
    aux=[]
    cont=0
    for j in [8,9,10,11,12,13,14,15,18]:
        a=np.load(path+Tip+'Red'+str(j)+'Input2.npy')
        aux.append(np.array(a))
        cont+=1
    sio.savemat(path+Tip+'inputs_None.mat',mdict={'inputs':aux})
    return aux
# def flujos(path,Tip):
#     aux=[]
#     cont=0
#     for j in [8,9,10,11,12,13,14,15,18]:
#         aux.append([])
#         for i in range(1,6):
#             a=sio.loadmat(path+Tip+'Flujos'+str(j)+str(i)+'.mat')['inputs']
#             for k in range(np.array(a).shape[0]):
#                 aux[cont].append(np.array(a[k]))
#         cont+=1
#     sio.savemat(path+Tip+'Flujos',mdict={'inputs':aux})
# def presiones(path,Tip):
#     aux=[]
#     cont=0
#     for j in [8,9,10,11,12,13,14,15,18]:
#         aux.append([])
#         for i in range(1,6):
#             a=sio.loadmat(path+Tip+'Presiones'+str(j)+str(i)+'.mat')['inputs']
#             for k in range(np.array(a).shape[0]):
#                 aux[cont].append(np.array(a[k]))
#         cont+=1
#     sio.savemat(path+Tip+'Presiones',mdict={'inputs':aux})

Mounted at /content/drive


# Objetivo 1

In [3]:
path='/content/drive/Shareddrives/red_gas_col/Data/S/'
Tip='MAPE/'
a=load_data(path,Tip)
Tip='MAE/'
a=load_data(path,Tip)

In [4]:
path='/content/drive/Shareddrives/red_gas_col/Data/M/'
Tip='MAPE/'
a1=load_data(path,Tip)
Tip='MAE/'
a2=load_data(path,Tip)

In [5]:
path='/content/drive/Shareddrives/red_gas_col/Data/D/'
Tip='MAPE/'
a1=load_data(path,Tip)
Tip='MAE/'
a2=load_data(path,Tip)

In [6]:
# path='/content/drive/Shareddrives/red_gas_col/Data/S/'
# Tip='Flujos/'
# flujos(path,Tip)
# path='/content/drive/Shareddrives/red_gas_col/Data/M/'
# flujos(path,Tip)
# path='/content/drive/Shareddrives/red_gas_col/Data/D/'
# flujos(path,Tip)

In [7]:
# path='/content/drive/Shareddrives/red_gas_col/Data/S/'
# Tip='Presiones/'
# presiones(path,Tip)
# path='/content/drive/Shareddrives/red_gas_col/Data/M/'
# presiones(path,Tip)
# path='/content/drive/Shareddrives/red_gas_col/Data/D/'
# presiones(path,Tip)